# 3.2 Velocity-based motion model

In this notebook we will explore **deterministic motion models** that, given a robot motion $\mathbf{u}$ and the previous robot pose $\mathbf{x}_{t-1}$, compute the new robot pose $\mathbf{x}_t$.

Depending on how the robot motion $\mathbf{u}$ is represented, two common formulations can be considered:

- **Velocity Motion Model:** the control input $\mathbf{u}_t$ is given by the robot velocities,
$$
\mathbf{u}_t =
\begin{bmatrix}
v_t \\
\omega_t
\end{bmatrix},
$$
where $v_t$ and $\omega_t$ denote the linear and angular velocities, respectively.  
Given the previous pose $\mathbf{x}_{t-1}$, the new pose is obtained through the velocity motion model:
$$
\mathbf{x}_t = g_v(\mathbf{x}_{t-1}, \mathbf{u}_t, \Delta t).
$$
Therefore, this formulation uses **two input components**: linear and angular velocity.

- **Odometry Motion Model:** instead of using velocities directly, the input is the **pose increment** estimated from the odometry readings between times $t-1$ and $t$,
$$
\Delta \mathbf{x}_t^{\mathrm{odo}} =
\begin{bmatrix}
\Delta x_t \\
\Delta y_t \\
\Delta \theta_t
\end{bmatrix}.
$$
This increment can be obtained from two consecutive odometry poses as
$$
\Delta \mathbf{x}_t^{\mathrm{odo}}
=
(\ominus \mathbf{x}_{t-1}^{\mathrm{odo}})
\oplus
\mathbf{x}_t^{\mathrm{odo}}.
$$
The robot pose is then updated according to
$$
\mathbf{x}_t =
g_o(\mathbf{x}_{t-1}, \Delta \mathbf{x}_t^{\mathrm{odo}}).
$$
In this case, the motion input contains **three components**: the increments in $x$, $y$, and orientation.

In summary:

<center><figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_motion/images/deterministic_motion_models.png" alt="">
</figure></center>

Concretely, in this notebook we will focus on the **Velocity-based** motion model.

## 3.2.1 The model


> ℹ️ **Usage:** The *velocity motion model* is mainly used for motion planning, where the details of the robot's movement are of importance and odometry information is not available (*e.g.* no wheel encoders are available).

This motion model is characterized by the use of two velocities to control the robot's movement: **linear velocity** $v$ and **angular velocity** $\omega$. The velocity motion model defines the function $g_v(\cdot)$ as:

$$g_v(\mathbf{x}_{t-1},\mathbf{u}_t,\Delta t) \, = \mathbf{x}_{t-1} \oplus \Delta \mathbf{x}_t(\mathbf{u}_t,\Delta t)$$

being $\Delta \mathbf{x}_t=[\Delta x_t, \Delta y_t, \Delta \theta_t]^T$ (assuming $\omega$ and $v$ constant):
- $\Delta x_t = \frac{v_t}{\omega_t} \sin(\omega_t\Delta t)$
- $\Delta y_t = \frac{v_t}{\omega_t} [1-\cos(\omega_t\Delta t)]$
- $\Delta \theta_t = \omega_t\Delta t$

Recall the curvature radius: $R = v_t/\omega_t$

The following figure illustrates this computation:

<center><figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_motion/images/differential_drive_kinematics.png" alt="" width="550px">
</figure></center>

In this way, this motion model is characterized by the following equations, depending on the value of the angular velocity $\omega$ (note that a division by zero would appear in the first case if $\omega=0$):

- If $\omega \neq 0$:

$$\begin{bmatrix} x_t \\ y_t \\ \theta_t \end{bmatrix} =
\begin{bmatrix} x_{t-1}+\Delta x_t\cos{\theta_{t-1}}-\Delta y_t \sin{\theta_{t-1}} \\
y_{t-1}+\Delta x_t \sin{\theta_{t-1}}+ \Delta y_t \cos{\theta_{t-1}} \\
\theta_{t-1} + \Delta \theta_t
\end{bmatrix} =
\begin{bmatrix} x_{t-1} \\ y_{t-1} \\ \theta_{t-1} \end{bmatrix} +
\begin{bmatrix} -R \sin \theta_{t-1} + R \sin(\theta_{t-1} + \Delta \theta_t) \\ R \cos \theta_{t-1} - R \cos(\theta_{t-1} + \Delta \theta_t) \\ \Delta \theta_t \end{bmatrix}$$

- If $\omega = 0$:

$$\begin{bmatrix} x_t \\ y_t \\ \theta_t \end{bmatrix} = \begin{bmatrix} x_{t-1} \\ y_{t-1} \\ \theta_{t-1} \end{bmatrix} + v_t \cdot \Delta t \begin{bmatrix} \cos \theta_{t-1} \\ \sin \theta_{t-1} \\ 0 \end{bmatrix}$$

In [ ]:
# In case of using Google Colab, mount Google Drive and enable interactive plots
try:
    from google.colab import drive, output
    drive.mount('/content/drive')
    # Change directory to this chapter's folder in your Drive copy of the course repo:
    %cd '/content/drive/MyDrive/Colab Notebooks/uma_robotics_2027/Chapter 03. Robot motion'
    %pip install -q ipympl
    # Make matplotlib detect the freshly installed ipympl backend:
    from matplotlib.backends.registry import backend_registry
    backend_registry.__init__()
    output.enable_custom_widget_manager()
except ImportError:
    pass

%matplotlib widget

# IMPORTS
import numpy as np
from numpy import random
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time


import sys
sys.path.append("..")
from utils.DrawRobot import DrawRobot
from utils.PlotEllipse import PlotEllipse

### **<span style="color:green"><b><i>ASSIGNMENT 1: The model in action</i></b></span>**

Modify and properly call the following `next_pose()` function, used in the `VelocityRobot` class below, which computes the next pose $\mathbf{x}_t$ of a robot given:
- its previous pose $\mathbf{x}_{t-1}$,
- the velocity movement command $\mathbf{u}_t=[v_t,\omega_t]^T$, and
- a lapse of time $\Delta t$.

Concretely, you have to complete the if-else statement that takes into account whether the robot moves in a straight line (so $\omega_t = 0$).
*Note: you don't have to modify the `None` in the function header nor in the `if cov is not None:` condition.*

<font color="blue"><i>Expected output:</i></font>

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/robotics_hub/main/robot_motion/images/fig3-2-1.png" alt="">
  <figcaption>Fig. 2: Route of our robot.</figcaption>
</figure>

In [ ]:
def next_pose(x, u, dt, cov=None):
    ''' This function takes pose x and transforms it according to the motion u=[v,w]'
        applying the differential drive model.

        Args:
            x: current pose
            u: differential command as a vector [v, w]'
            dt: Time interval in which the movement occurs
            cov: covariance of our movement. If not None, then add gaussian noise
    '''
    if cov is not None:
        u += np.sqrt(cov) @ random.randn(2, 1)
        #u = np.random.multivariate_normal(u.flatten(),cov)

    if u[1] == 0: #linear motion w=0
        next_x = np.vstack([None,
                       None,
                       None])
    else: #Non-linear motion w=!0
        R = u[0]/u[1] #v/w=r is the curvature radius
        next_x = np.vstack([None,
                       None,
                       None])

    return next_x

In [ ]:
class VelocityRobot(object):
    """ Mobile robot implementation that uses velocity commands.

        Attr:
            pose: expected pose of the robot in the real world (without taking account noise)
            dt: Duration of each step in seconds
    """
    def __init__(self, pose, dt):
        self.pose = pose
        self.dt = dt

    def step(self, u):
        self.pose = next_pose(None, None, None)

    def draw(self, fig, ax):
        DrawRobot(fig, ax, self.pose)

**Test the movement of your robot** using the demo below.

In [ ]:
def main(robot, nSteps):

    v = 1 # Linear Velocity
    l = 0.5 #Half the width of the robot

    # MATPLOTLIB
    fig, ax = plt.subplots()
    plt.ion()
    fig.canvas.draw()
    plt.xlim((-2, 20))
    plt.ylim((-2, 30))
    plt.fill([7.5, 7.5, 16, 16, 20, 20],[-2, 7.5, 7.5, 20, 20, -2],
             facecolor='lightgray', edgecolor='gray', linewidth=3)
    plt.fill([-3, 0, 0, 7.5, 7.5, -3],[5, 5, 17.5, 17.5, 32, 32],
             facecolor='lightgray', edgecolor='gray', linewidth=3)

    plt.grid()

    # MAIN LOOP
    for k in range(1, nSteps + 1):
        #control is a wiggle with constant linear velocity
        u = np.vstack((v, np.pi / 10 * np.sin(4 * np.pi * k/nSteps)))

        robot.step(u)

        #draw occasionally
        if (k-1)%20 == 0:
            robot.draw(fig, ax)
            clear_output(wait=True)
            display(fig)
            time.sleep(0.1)

    plt.close()


In [ ]:
# RUN
dT = 0.1 # time steps size
pose = np.vstack([0., 0., 0.])

robot = VelocityRobot(pose, dT)
main(robot, nSteps=400)

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Now that you have some experience with robot motion and the velocity motion model, **answer the following questions**:

- Why do we need to consider two different cases when applying the $g_v(\cdot)$ function, that is, calculating the new robot pose?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- How many parameters compound the motion command $\mathbf{u}_t$ in this model?

    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

## <font color="orange"><b><i>AI Appendix</i></b></font>

### A.1 How I used AI
- **Model/tool:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **What I asked AI to help with (bullets):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **Best prompt I used (paste):**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **What I kept vs. changed from the AI output:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

### A.2 AI review
Ask one generative AI to review your memo using this prompt:

```
Act as a technical reviewer of this robotics notebook. Read my memo and return exactly 5 bullet points:

- two on theory (key ideas),
- two on practice (relevant programming concepts of python used, not particular functions or variables),
- one on how to improve the code (most critical aspect: limitations, parameter choices, runtime/robustness, possible bugs).

Keep each bullet ≤15 words and make them specific to my work.
```

Paste its **5** items here:

- **(Theory 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Theory 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 1)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Practice 2)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
- **(Improvement)** AI claim:
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

Finally, **do you agree with them?**, **would you have selected others?**
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>